In [ ]:
import os
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings("ignore")

In [ ]:
def nearest_1d_datetime(time_src, y_src, time_tgt, max_gap="10min"):
    """
    Nearest-neighbor sampling of y_src(time_src) onto time_tgt.

    Drops NaT in time_src and NaN/inf in y_src.
    Returns NaN for targets whose nearest observation is farther than max_gap.
    """
    time_src = pd.to_datetime(time_src)
    time_tgt = pd.to_datetime(time_tgt)
    tol_ns = pd.Timedelta(max_gap).value  # nanoseconds

    m = (~pd.isna(time_src)) & np.isfinite(y_src)
    if np.count_nonzero(m) < 1:
        return np.full(len(time_tgt), np.nan, dtype=float)

    xs = time_src[m].view("i8")
    ys = np.asarray(y_src[m], dtype=float)

    # sort by time
    order = np.argsort(xs)
    xs = xs[order]
    ys = ys[order]

    xt = time_tgt.view("i8")
    idx = np.searchsorted(xs, xt)

    # nearest neighbor index for each target
    nn = np.empty(len(xt), dtype=int)

    # handle left edge
    left_edge = idx == 0
    nn[left_edge] = 0

    # handle right edge
    right_edge = idx == len(xs)
    nn[right_edge] = len(xs) - 1

    # interior: decide whether left or right is closer
    mid = (~left_edge) & (~right_edge)
    left_i = idx[mid] - 1
    right_i = idx[mid]

    left_dist = xt[mid] - xs[left_i]
    right_dist = xs[right_i] - xt[mid]

    use_right = right_dist < left_dist
    nn[mid] = np.where(use_right, right_i, left_i)

    out = ys[nn]

    # apply tolerance
    nearest_dist = np.abs(xt - xs[nn])
    out[nearest_dist > tol_ns] = np.nan

    return out

In [ ]:
project = "ArctSum2025"   # <-- set this (or read from elsewhere)

out_dir = "/home/maltem/work/python/data/"+project+"/cf_out"
os.makedirs(out_dir, exist_ok=True)

INFILE = '/home/maltem/work/python/data/ArctSum2025/2025_KVS_ArctSum_withpositions_with_tempcalibration_cleaned.nc'
ds = xr.open_dataset(INFILE, decode_times=True)

#--- A look-up file giving the information on the first sensor in ice and the ice-thickness 
#--- for each individual buoy
file = "/home/maltem/work/python/data/ArctSum2025/Deploymnent_ArctSum_Aug_2025.csv"
df = pd.read_csv(file, header=1)
df.columns = df.columns.str.strip()

#--- Loop over buoys :

for tr in range(len(ds['trajectory'])):
    BuoyID = str(ds["trajectory"][tr].values)

    ice_thickness = float(df.loc[df["BuoyID"] == BuoyID, "Icethickness"].iloc[0])
    sensorID      = int(df.loc[df["BuoyID"] == BuoyID, "sensorIce2"].iloc[0])

    print("Ice Thickness:", ice_thickness)
    print("BuoyID:", BuoyID)
    print("First sensor in sea-ice:", sensorID)

    # Project time axis on common time-axis from the lat/lon

    # ----- Time Temperature -----
    time_temp = pd.to_datetime(ds["time_temp"][tr, :].values)
    valid_temp = ~pd.isna(time_temp)
    time_temp = np.asarray(time_temp[valid_temp]) 

    # ----- Time Position -----
    time = pd.to_datetime(ds["time"][tr, :].values)
    valid_pos= ~pd.isna(time)
    time = np.asarray(time[valid_pos])

    # ----- Time Position -----
    time_wave = pd.to_datetime(ds["time_waves_imu"][tr, :].values)
    valid_wave= ~pd.isna(time_wave)
    time_wave = np.asarray(time_wave[valid_wave])

    #--- GPS positions
    lat  = ds["lat"].isel(trajectory=tr).values[:]
    lon  = ds["lon"].isel(trajectory=tr).values[:]
    lat  = np.asarray(lat[valid_pos])
    lon  = np.asarray(lon[valid_pos])

    # ----- Tair at string position 27 (index 26) -----
    air_idx = np.where(ds['string_position'].values == 'air')[0]
    print("Position of air temperature in the string: ",air_idx)
    Tair_all = ds["temperature_calibrated_at_positions"].isel(trajectory=tr).values[:, air_idx]
    Tair = np.asarray(Tair_all[valid_temp]).squeeze()  # 1D numpy array

    # ----- String temperatures (first 26 sensors) -----
    T_all = ds["temperature_calibrated_at_positions"].isel(trajectory=tr).values[:, 0:len(ds['string_position'])-1]
    Tstring = T_all[valid_temp, :]

    # ---- Skin temperature Infra Red sensor ----
    Tskin_all= ds["temp_mlx_ir"].isel(trajectory=tr).values
    Tskin = np.asarray(Tskin_all[valid_temp]).squeeze()  # 1D numpy array

    # ---- Wave variables ---- #

    Hs0_all = ds["pHs0"].isel(trajectory=tr).values
    Hs0 = np.asarray(Hs0_all[valid_wave]).squeeze()  # 1D numpy array
    T02_all = ds["pT02"].isel(trajectory=tr).values
    T02 = np.asarray(T02_all[valid_wave]).squeeze()  # 1D numpy array
    T24_all = ds["pT24"].isel(trajectory=tr).values
    T24 = np.asarray(T24_all[valid_wave]).squeeze()  # 1D numpy array
    y0 = ds["string_position"].values[0:len(ds['string_position'])-1].astype(float)
    y = (y0 - y0[sensorID - 1]).astype(float)
    print(y)

    Tair_on_time  = nearest_1d_datetime(time_temp, Tair, time, max_gap="10min")
    Tskin_on_time = nearest_1d_datetime(time_temp, Tskin, time, max_gap="10min")
    Tstring_on_time = np.full((len(time), Tstring.shape[1]), np.nan, dtype=float)

    for j in range(Tstring.shape[1]):
        Tstring_on_time[:, j] = nearest_1d_datetime(
            time_temp, Tstring[:, j], time, max_gap="10min"
        )
    
    Hs0_on_time = nearest_1d_datetime(time_wave, Hs0, time, max_gap="30min")
    T02_on_time = nearest_1d_datetime(time_wave, T02, time, max_gap="30min")
    T24_on_time = nearest_1d_datetime(time_wave, T24, time, max_gap="30min")
    # ---- Write CF-compliant netcdf file --- #
    


    out_file = os.path.join(out_dir, f"{BuoyID}_hourly_cf.nc")

    # -----------------------------
    convert_to_kelvin = True

    def _toK(x):
        x = np.asarray(x, dtype=float)
        return x + 273.15 if convert_to_kelvin else x

    # -----------------------------
    # Ensure time is datetime64[ns]
    # -----------------------------
    time_dt = pd.to_datetime(time).to_numpy()
    epoch = np.datetime64("1970-01-01T00:00:00", "ns")
    time_sec = (time_dt - epoch).astype("timedelta64[ns]").astype(np.float64) / 1e9  # float seconds

    # -----------------------------
    # Ensure depth sign convention: positive down
    y_depth = np.asarray(y, dtype=float)

    # -----------------------------
    # Build the CF/ACDD dataset
    # -----------------------------
    ds_out = xr.Dataset(
        coords={
            "time": ("time", time_sec.astype(np.float64)),
            "depth": ("depth", y_depth.astype(np.float32)),
        }
    )
    
    # Coordinates / core geolocation
    ds_out["latitude"]  = xr.DataArray(np.asarray(lat, dtype=float), dims=("time",))
    ds_out["longitude"] = xr.DataArray(np.asarray(lon, dtype=float), dims=("time",))

    ds_out["latitude"].attrs.update(
        standard_name="latitude",
        units="degrees_north",
    )
    ds_out["longitude"].attrs.update(
        standard_name="longitude",
        units="degrees_east",
    )

    ds_out["time"].attrs.update(
        standard_name="time",
        long_name="Time",
        units="seconds since 1970-01-01 00:00:00",
        calendar="gregorian",
        axis="T",
    )

  
    ds_out["depth"].attrs.update(
        standard_name="depth",
        long_name="Depth relative to first sensor embedded in sea ice",
        units="m",
        positive="down",
        axis="Z",
    )

    # Air temperature
    ds_out["air_temperature"] = xr.DataArray(_toK(Tair_on_time), dims=("time",))
    ds_out["air_temperature"].attrs.update(
        standard_name="air_temperature",
        long_name="Air temperature at 1 m",
        units="K",
        cell_methods="time: point",
        coordinates="time latitude longitude",
    )

    # Skin temperature (IR)
    ds_out["surface_temperature"] = xr.DataArray(_toK(Tskin_on_time), dims=("time",))
    ds_out["surface_temperature"].attrs.update(
        standard_name="surface_temperature",
        long_name="Surface (skin) temperature",
        units="K",
        cell_methods="time: point",
        coordinates="time latitude longitude",
    )

    # Sea-ice temperature along string
    ds_out["sea_ice_temperature"] = xr.DataArray(_toK(Tstring_on_time), dims=("time", "depth"))
    ds_out["sea_ice_temperature"].attrs.update(
        standard_name="sea_ice_temperature",
        long_name="Sea-ice temperature along sensor string",
        units="K",
        cell_methods="time: point",
        coordinates="time latitude longitude depth",
    )

    # Waves
    ds_out["significant_wave_height"] = xr.DataArray(np.asarray(Hs0_on_time, dtype=float), dims=("time",))
    ds_out["significant_wave_height"].attrs.update(
        standard_name="sea_surface_wave_significant_height",
        long_name="Significant wave height",
        units="m",
        cell_methods="time: point",
        coordinates="time latitude longitude",
    )

    ds_out["wave_period_t02"] = xr.DataArray(np.asarray(T02_on_time, dtype=float), dims=("time",))
    ds_out["wave_period_t02"].attrs.update(
        long_name="Mean wave period T02",
        units="s",
        cell_methods="time: point",
        coordinates="time latitude longitude",
    )

    ds_out["wave_period_t24"] = xr.DataArray(np.asarray(T24_on_time, dtype=float), dims=("time",))
    ds_out["wave_period_t24"].attrs.update(
        long_name="Mean wave period T24",
        units="s",
        cell_methods="time: point",
        coordinates="time latitude longitude",
    )

    
    # -----------------------------
    # Global attributes (CF/ACDD)
    # -----------------------------
    t0 = pd.to_datetime(time_dt[0]).isoformat()
    t1 = pd.to_datetime(time_dt[-1]).isoformat()

    ds_out.attrs.update(
        Conventions="CF-1.10 ACDD-1.3",
        featureType="trajectory",
        title=f"{project} - buoy hourly product (single trajectory, CF compliant)",
        summary=(
            f"Instantaneous hourly product for drifting OpenMetBuoy trajectory from the {project} campaign. "
            "Data are projected onto a shared hourly time axis using nearest-neighbour matching with a time tolerance."
        ),
        keywords=f"buoy, Arctic, waves, sea ice, meteorology, {project}",
        project=project,
        references="https://openmetbuoy-arctic.com/datarelease.html",
        source="Hourly nearest-neighbour projection onto time (10 min tol for T/pos; 30 min for Hs/T02).",
        creator_name="Malte Müller",
        creator_email="maltem@met.no",
        creator_institution="Norwegian Meteorological Institute",
        trajectory_id=str(BuoyID),
        history=f"Created {pd.Timestamp.utcnow().isoformat()} by write_one_buoy_cf()",
        time_coverage_start=t0,
        time_coverage_end=t1,
        geospatial_lat_min=float(np.nanmin(ds_out["latitude"].values)),
        geospatial_lat_max=float(np.nanmax(ds_out["latitude"].values)),
        geospatial_lon_min=float(np.nanmin(ds_out["longitude"].values)),
        geospatial_lon_max=float(np.nanmax(ds_out["longitude"].values)),
    )

    # -----------------------------
    # Encoding: CF time + compression
    # ------------
#    
    encoding = {
        "time": {"dtype": "float64"},  
        "latitude": {"zlib": True, "complevel": 4, "_FillValue": np.nan},
        "longitude": {"zlib": True, "complevel": 4, "_FillValue": np.nan},
        "air_temperature": {"zlib": True, "complevel": 4, "_FillValue": np.nan},
        "surface_temperature": {"zlib": True, "complevel": 4, "_FillValue": np.nan},
        "sea_ice_temperature": {"zlib": True, "complevel": 4, "_FillValue": np.nan},
        "significant_wave_height": {"zlib": True, "complevel": 4, "_FillValue": np.nan},
        "wave_period_t02": {"zlib": True, "complevel": 4, "_FillValue": np.nan},
        "wave_period_t24": {"zlib": True, "complevel": 4, "_FillValue": np.nan},
    }

    # Remove conflicting serialization keys from attrs
    #ds_out["time"].attrs.pop("units", None)
    #ds_out["time"].attrs.pop("calendar", None)

   
    # Write
    ds_out.to_netcdf(out_file, encoding=encoding)
    print("Wrote:", out_file)    